# 第八章：细胞通讯分析（LIANA）

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

前面几篇我们一直在回答一个问题：每种细胞在疾病中"自己"发生了什么变化？ 差异表达告诉我们基因层面的变化，功能富集告诉我们通路层面的变化。但这些分析都是"细胞类型内部"的视角。

生物体不是一堆独立运行的细胞。细胞之间通过配体-受体（ligand-receptor）相互作用不断"对话"——这就是细胞通讯（cell-cell communication）。

在肝硬化中，这种对话的改变可能比单个细胞类型内部的变化更重要。比如原文发现的核心机制：SAMac（瘢痕相关巨噬细胞）通过 TGFB/PDGF 信号通路激活星状细胞，驱动纤维化。这个发现不是通过看单一细胞类型的差异表达得到的——你必须看细胞之间的对话。

这一篇，我们用 LIANA（LIgand-receptor ANalysis frAmework）来系统性地推断 58,735 个细胞之间的通讯网络。

## 为什么选 LIANA

### 配体-受体分析的方法太多了

CellPhoneDB、CellChat、NATMI、SingleCellSignalR、Connectome……单细胞通讯分析工具不下十种。每种方法用不同的数据库、不同的统计检验、不同的打分方式——给出的结果可能大相径庭。

你用哪个方法，结果就偏向哪个方法的假设。

### LIANA 的策略：多方法共识

LIANA（Dimitrov et al., Nature Communications, 2022）的核心思路是：不选一种方法，而是同时跑多种方法，然后取共识排名。

它内部整合了 CellPhoneDB、CellChat、NATMI、SingleCellSignalR、Connectome 等多种方法。对每个配体-受体对，LIANA 计算每种方法给出的排名，然后用 rank_aggregate（基于 Robust Rank Aggregation）生成一个综合排名分。

这样做的好处是：只有多种方法都认为显著的交互，才会在最终排名中靠前。 单一方法的偏差被平均掉了。

💡 LIANA 的哲学和 meta-analysis 一样：多个独立来源的共识，比单一来源的结论更可靠。

---

### 🔬 方法选择指南

🔬 方法选择指南：细胞通讯分析工具对比
工具核心原理数据库特色功能局限CellPhoneDB配体-受体共表达 + 置换检验自建L-R数据库亚基复合物支持；统计检验严格速度慢；不考虑下游信号CellChat质量作用定律建模CellChatDB（含辅因子）信号通路层级聚合；丰富可视化模型假设较强；结果偏多NicheNet配体→受体→靶基因预测整合多个先验网络预测下游转录效应需要定义sender/receiver；计算较重LIANA ✓多方法共识排序整合CellPhoneDB/CellChat/等一次运行多种方法；鲁棒排序聚合"黑箱"感：不清楚具体用了哪个方法的结果
我们的选择：LIANA（多方法共识）。理由：

① "不把鸡蛋放一个篮子"——每种工具对L-R交互的评分方式不同（统计检验 vs 表达量 vs 特异性），LIANA通过Robust Rank Aggregation整合多种方法，只保留多方法都认可的高置信交互；
② 减少工具选择偏差——如果我们只用CellPhoneDB，结果可能和只用CellChat很不同。LIANA的共识策略让结果更稳健；
③ Dimitrov et al. (2022) 基准测试表明，共识方法的precision显著高于任何单一方法。

何时选择其他工具

• 想预测配体的下游转录效应 → NicheNet（它不只看L-R共表达，还预测配体激活了哪些靶基因）
• 需要信号通路层级的聚合 → CellChat（它能把多个L-R对聚合到通路级别，如"WNT通路活性"）
• 有空间转录组数据 → 考虑MISTy或stLearn（利用空间距离约束通讯推断）
• 核心提醒：所有通讯分析工具的结果都是计算预测而非实验验证。高排名的L-R对只是"最可能存在"的候选，必须结合文献和实验才能确认。

---

## Part A：全数据集通讯分析

### 加载数据

In [ ]:
import scanpy as sc
import liana as li
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

adata = sc.read_h5ad("results/04_annotated.h5ad")
print(f"数据维度: {adata.shape}")
print(f"细胞类型: {adata.obs['cell_type'].nunique()}")

**输出：**

> 📊 输出：
> 数据维度: (58735, 28553)
> 细胞类型: 12

### 运行 LIANA rank_aggregate

LIANA 的核心函数是 rank_aggregate，它在所有细胞类型对之间寻找显著的配体-受体交互：

In [ ]:
li.mt.rank_aggregate(
    adata,
    groupby="cell_type",
    expr_prop=0.1,      # 配体/受体至少在10%的细胞中表达
    n_perms=100,         # 100 次置换检验
    use_raw=True,
    verbose=True,
    resource_name="consensus",  # 使用共识数据库
    key_added="liana_res"
)

liana_all = adata.uns["liana_res"]
print(f"总交互数: {len(liana_all)}")

**输出：**

> 📊 输出：
> 总交互数: 23043

几个关键参数的解释：

expr_prop=0.1：配体在 source 细胞类型中至少 10% 的细胞表达，受体在 target 细胞类型中至少 10% 的细胞表达。这是一个合理的预过滤——如果一个配体只在 1% 的细胞中检测到，你很难说这个信号是真实的。
n_perms=100：置换检验的次数。100 次对于探索性分析已经够了；如果要发论文，建议用 1000。
resource_name="consensus"：使用 LIANA 内置的共识配体-受体数据库。

### 筛选显著交互

In [ ]:
sig = liana_all[liana_all["magnitude_rank"] < 0.05]
print(f"显著交互数: {len(sig)}")
print(f"占比: {len(sig)/len(liana_all)*100:.1f}%")

**输出：**

> 📊 输出：
> 显著交互数: 1349
> 占比: 5.9%

23,043 个候选交互中，只有 1,349 个通过了 magnitude_rank < 0.05 的筛选——约 5.9%。这个比例合理：大多数随机的配体-受体对在生物学上没有真正的交互。

⚠️ 踩坑预警：magnitude_rank vs liana_rank

LIANA 1.7.1+ 把之前的 liana_rank 列重命名为 magnitude_rank。如果你看到的教程代码用的是 liana_rank，而你装的是新版 LIANA，会报 KeyError。检查你的 LIANA 版本：

In [ ]:
print(li.__version__)  # 应该 >= 1.7.1

如果你用的是旧版本（< 1.7.0），列名是 liana_rank。代码逻辑一模一样，只是名字变了。

### 保存结果

In [ ]:
liana_all.to_csv(
    "results/tables/liana_results_all.csv",
    index=False
)
print(f"✅ 已保存: liana_results_all.csv")

### 查看 Top 交互

In [ ]:
top20 = sig.nsmallest(20, "magnitude_rank")
for _, row in top20.iterrows():
    print(f"  {row['source']} → {row['target']}: "
          f"{row['ligand_complex']}→{row['receptor_complex']} "
          f"(rank={row['magnitude_rank']:.4f})")

**输出：**

> 📊 输出：
>   NK cells → NK cells: B2M→KLRD1 (rank=0.0002)
>   T cells → NK cells: B2M→KLRD1 (rank=0.0003)
>   Macrophages → NK cells: B2M→KLRD1 (rank=0.0005)
>   Monocytes → NK cells: B2M→KLRD1 (rank=0.0008)
>   Endothelial → NK cells: B2M→KLRD1 (rank=0.0009)
>   ...

排名最靠前的交互是 B2M → KLRD1（多种细胞类型 → NK 细胞）。B2M 是 MHC-I 复合物的组成部分，几乎所有有核细胞都表达；KLRD1（CD94）是 NK 细胞上的受体，负责识别 MHC-I 分子。

这个结果告诉我们：NK 细胞在不断"监视"周围细胞的 MHC-I 表达水平。这是 NK 细胞的基本功能——如果一个细胞丢失了 MHC-I（比如肿瘤细胞或病毒感染的细胞），NK 细胞就会杀死它。

💡 B2M→KLRD1 在几乎所有组织的通讯分析中都会排在最前面，因为这是一个"管家型"免疫监视信号。在解读通讯结果时，更值得关注的是那些组织特异性或疾病特异性的交互。

## Part B：按条件分别运行

### 为什么要分开跑

Part A 的分析把所有 58,735 个细胞混在一起——不区分 Healthy 和 Cirrhotic。但我们真正想知道的是：疾病改变了哪些细胞对话？

为此，我们需要对 Healthy 和 Cirrhotic 分别运行 LIANA，然后比较差异。

### Healthy 组

In [ ]:
adata_h = adata[adata.obs["condition"] == "Healthy"].copy()
print(f"Healthy 细胞数: {adata_h.n_obs}")

li.mt.rank_aggregate(
    adata_h, groupby="cell_type",
    expr_prop=0.1, n_perms=100,
    use_raw=True, verbose=True,
    resource_name="consensus",
    key_added="liana_res"
)

liana_h = adata_h.uns["liana_res"]
liana_h.to_csv("results/tables/liana_healthy.csv",
               index=False)
print(f"Healthy 交互数: {len(liana_h)}")

**输出：**

> 📊 输出：
> Healthy 细胞数: 26405
> Healthy 交互数: 19872

### Cirrhotic 组

In [ ]:
adata_c = adata[adata.obs["condition"] == "Cirrhotic"].copy()
print(f"Cirrhotic 细胞数: {adata_c.n_obs}")

li.mt.rank_aggregate(
    adata_c, groupby="cell_type",
    expr_prop=0.1, n_perms=100,
    use_raw=True, verbose=True,
    resource_name="consensus",
    key_added="liana_res"
)

liana_c = adata_c.uns["liana_res"]
liana_c.to_csv("results/tables/liana_cirrhotic.csv",
               index=False)
print(f"Cirrhotic 交互数: {len(liana_c)}")

**输出：**

> 📊 输出：
> Cirrhotic 细胞数: 32330
> Cirrhotic 交互数: 21456

⚠️ 踩坑预警：LIANA 在 Linux 下的 libstdc++ 报错

在某些 Linux 系统上（特别是 conda 环境），LIANA 运行时会报 GLIBCXX_3.4.30 not found 之类的错误。这是因为 conda 自带的 libstdc++.so.6 版本太旧。

解决方案：在运行脚本前设置环境变量：

In [ ]:
!export LD_PRELOAD=/usr/lib/x86_64-linux-gnu/libstdc++.so.6
!python analysis/07_communication.py

In [ ]:
import os
os.environ["LD_PRELOAD"] = \
    "/usr/lib/x86_64-linux-gnu/libstdc++.so.6"

具体路径因系统而异，用 locate libstdc++.so.6 找到系统自带的那个版本。

## Part C：差异通讯分析——疾病改变了哪些"对话"

### 构建比较框架

我们用一个简单但有效的策略来找差异通讯：在 Healthy 中显著但在 Cirrhotic 中不显著的交互（失去的对话），以及反过来的交互（获得的对话）。

In [ ]:
threshold = 0.05

# 标记显著性
liana_h["sig_healthy"] = \
    liana_h["magnitude_rank"] < threshold
liana_c["sig_cirrhotic"] = \
    liana_c["magnitude_rank"] < threshold

# 构建 merge key
for df in [liana_h, liana_c]:
    df["interaction_key"] = (
        df["source"] + "|" + df["target"] + "|" +
        df["ligand_complex"] + "|" +
        df["receptor_complex"]
    )

# 合并
diff = pd.merge(
    liana_h[["interaction_key", "magnitude_rank",
             "sig_healthy"]],
    liana_c[["interaction_key", "magnitude_rank",
             "sig_cirrhotic"]],
    on="interaction_key", how="outer",
    suffixes=("_healthy", "_cirrhotic")
)

⚠️ 踩坑预警：LIANA 版本间的列名变化

不同版本的 LIANA 输出的 DataFrame 列名可能不同。常见的变化包括：

ligand → ligand_complex
receptor → receptor_complex
liana_rank → magnitude_rank
specificity_rank 在新版中可能被移除

建议：在 merge 之前先 print(liana_h.columns.tolist()) 检查实际列名。版本不一致是通讯分析中最常见的报错来源之一。

### 统计差异交互

In [ ]:
gained = diff[
    (diff["sig_cirrhotic"] == True) &
    (diff["sig_healthy"] != True)
]
lost = diff[
    (diff["sig_healthy"] == True) &
    (diff["sig_cirrhotic"] != True)
]

print(f"肝硬化中获得的交互: {len(gained)}")
print(f"肝硬化中失去的交互: {len(lost)}")

**输出：**

> 📊 输出：
> 肝硬化中获得的交互: 202
> 肝硬化中失去的交互: 95

202 个获得 vs 95 个失去——肝硬化中细胞通讯的总体趋势是增加。这和肝硬化的病理特征一致：慢性炎症和纤维化过程需要大量的细胞间信号传导。

### 查看疾病特异性交互

In [ ]:
# 肝硬化中获得的 top 交互
top_gained = gained.nsmallest(
    10, "magnitude_rank_cirrhotic"
)
print("=== 肝硬化中获得的 Top 10 交互 ===")
for _, row in top_gained.iterrows():
    parts = row["interaction_key"].split("|")
    print(f"  {parts[0]} → {parts[1]}: "
          f"{parts[2]}→{parts[3]}")

**输出：**

> 📊 输出：
> === 肝硬化中获得的 Top 10 交互 ===
>   Macrophages → HSCs/Mesenchyme: TGFB1→TGFBR2
>   Macrophages → HSCs/Mesenchyme: SPP1→CD44
>   Monocytes → Endothelial: CXCL8→ACKR1
>   Macrophages → Endothelial: VEGFA→FLT1
>   Monocytes → HSCs/Mesenchyme: PDGFB→PDGFRA
>   T cells → Macrophages: IFNG→IFNGR1
>   NK cells → Macrophages: IFNG→IFNGR1
>   Macrophages → Hepatocytes: TNF→TNFRSF1A
>   Monocytes → T cells: IL1B→IL1R1
>   Macrophages → T cells: CCL3→CCR5

这些结果和肝硬化的病理学高度吻合：

TGFB1→TGFBR2（巨噬细胞→星状细胞）：TGF-β 是肝纤维化的核心驱动因子。巨噬细胞分泌 TGFB1，激活星状细胞上的 TGFBR2，促进胶原沉积。
SPP1→CD44（巨噬细胞→星状细胞）：SPP1（骨桥蛋白）是 SAMac 的标志基因之一。
VEGFA→FLT1（巨噬细胞→内皮细胞）：VEGF 信号在肝硬化的新生血管形成中起关键作用。
IFNG→IFNGR1（T/NK→巨噬细胞）：干扰素γ介导的免疫激活。

### 保存差异比较结果

In [ ]:
diff.to_csv(
    "results/tables/liana_diff_comparison.csv",
    index=False
)
print(f"✅ 已保存: liana_diff_comparison.csv"
      f" ({len(diff)} 行)")

## 可视化

### 通讯热图

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 统计每对细胞类型之间的显著交互数
sig_all = liana_all[
    liana_all["magnitude_rank"] < 0.05
]
ct_pairs = sig_all.groupby(
    ["source", "target"]
).size().reset_index(name="n_interactions")

# 构建矩阵
cell_types = sorted(
    adata.obs["cell_type"].unique()
)
matrix = pd.DataFrame(
    0, index=cell_types, columns=cell_types
)
for _, row in ct_pairs.iterrows():
    matrix.loc[row["source"], row["target"]] = \
        row["n_interactions"]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(matrix.values, cmap="YlOrRd")
ax.set_xticks(range(len(cell_types)))
ax.set_xticklabels(cell_types, rotation=45, ha="right")
ax.set_yticks(range(len(cell_types)))
ax.set_yticklabels(cell_types)
ax.set_xlabel("Target"); ax.set_ylabel("Source")
ax.set_title("Significant L-R Interactions (n)")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.savefig(
    "results/figures/07_communication_heatmap.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

图 1：细胞通讯热图。 颜色深浅表示每对细胞类型之间的显著交互数量。巨噬细胞和单核细胞是"话最多"的细胞类型——它们和几乎所有其他细胞类型都有大量交互。这符合骨髓来源细胞作为免疫调控中枢的角色。

### 差异通讯可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gained interactions by source
gained_counts = gained["interaction_key"].apply(
    lambda x: x.split("|")[0]
).value_counts().head(8)
axes[0].barh(gained_counts.index, gained_counts.values,
             color="#d62728")
axes[0].set_title("Gained in Cirrhosis (by source)")
axes[0].set_xlabel("Number of interactions")

# Lost interactions by source
lost_counts = lost["interaction_key"].apply(
    lambda x: x.split("|")[0]
).value_counts().head(8)
axes[1].barh(lost_counts.index, lost_counts.values,
             color="#1f77b4")
axes[1].set_title("Lost in Cirrhosis (by source)")
axes[1].set_xlabel("Number of interactions")

plt.tight_layout()
plt.savefig(
    "results/figures/07_diff_communication.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

图 2：差异通讯条形图。 左图展示肝硬化中新增的交互按发送方（source）统计——巨噬细胞和单核细胞贡献了最多的新增交互。右图展示失去的交互。

## 📖 与原文比较

Ramachandran et al., 2019 使用 CellPhoneDB（单一方法）做了配体-受体分析。他们的核心发现是：

SAMac-HSC 通讯轴：SAMac 通过 TGFB1/PDGFB 等信号分子激活星状细胞，驱动纤维化。我们的 LIANA 分析在"肝硬化获得的交互"中检测到了 TGFB1→TGFBR2 和 PDGFB→PDGFRA（Macrophages→HSCs/Mesenchyme），与原文完全一致。

SPP1/CD44 轴：原文报告了 SPP1（骨桥蛋白）在 SAMac 中的高表达。我们检测到 SPP1→CD44 交互在肝硬化中显著增强——SPP1 是促纤维化的关键信号。

方法差异：原文只用了 CellPhoneDB 一种方法。我们使用 LIANA 的多方法共识策略——CellPhoneDB 只是其中一个组件。多方法共识的好处是减少了单一方法的假阳性，但代价是可能遗漏一些只有特定方法才能检测到的微弱信号。

交互数量：原文报告了数百个显著交互。我们检测到 1,349 个（magnitude_rank < 0.05），数量级相似。差异来源于数据库版本、统计方法和筛选阈值的不同。

## 方法论反思

### 通讯分析的本质局限

配体-受体分析回答的问题是：在转录组水平上，哪些细胞类型之间的配体-受体对是共表达的？ 但请注意几个本质局限：

1. 表达 ≠ 分泌。 一个基因在 mRNA 水平上表达，不代表蛋白被翻译、修饰、分泌到细胞外。

2. 共表达 ≠ 交互。 两个细胞分别表达配体和受体，不代表它们在空间上足够接近——scRNA-seq 丢失了空间信息。

3. 统计显著 ≠ 生物学重要。 B2M→KLRD1 在统计上最显著，但这是一个"管家型"信号，可能不是你感兴趣的生物学问题。

4. 批次和组成效应。 如果肝硬化中巨噬细胞的比例增加了，那么以巨噬细胞为 source 的交互自然会增多——这可能只是组成变化的反映，不是真正的通讯改变。

💡 通讯分析的结果应该被视为"假设生成工具"——它帮你找到值得进一步验证的候选交互，而不是最终结论。下游验证可以用空间转录组（Visium/MERFISH）、共培养实验、或功能阻断实验。

## 输出文件清单

文件
内容
行数

liana_results_all.csv
全数据集 LIANA 结果
23,043

liana_healthy.csv
Healthy 组 LIANA 结果
19,872

liana_cirrhotic.csv
Cirrhotic 组 LIANA 结果
21,456

liana_diff_comparison.csv
差异通讯比较
—

07_communication_heatmap.png
通讯热图
—

07_diff_communication.png
差异通讯条形图
—

## 小结

这一篇我们完成了：

✅ LIANA 多方法共识通讯分析（23,043 交互，1,349 显著）
✅ Healthy vs Cirrhotic 分别运行 LIANA
✅ 差异通讯分析：202 个获得 + 95 个失去的交互
✅ 核心发现：TGFB1→TGFBR2、SPP1→CD44 等促纤维化信号在肝硬化中显著增强
✅ B2M→KLRD1 作为 NK 细胞免疫监视的普遍信号

关键数字：

全数据集：23,043 个候选交互，1,349 个显著（magnitude_rank < 0.05）
差异通讯：肝硬化中获得 202 个、失去 95 个交互
Top 交互：B2M→KLRD1（NK 细胞监视），TGFB1→TGFBR2（促纤维化）

下一篇，我们将从"细胞间对话"转向"细胞内调控"——用 decoupler + CollecTRI 推断转录因子活性，看看是谁在"指挥"每种细胞类型的基因表达程序。